# PodcastGen-Agent
Generate full podcast episodes from a topic using AI.

**Before you start**
1. Runtime → Change runtime type → **GPU** (T4 is fine)
2. Run every cell in order from top to bottom
3. If install fails, use **Runtime → Restart session**, then re-run from cell 1

In [ ]:
# Colab setup (self-contained; runs before repo clone)
# Warnings about gradio/langchain can be ignored.

import subprocess
import sys

HF_STACK = ("transformers==4.46.3", "tokenizers==0.20.3")

def pip(*packages, extra=None):
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if extra:
        cmd.extend(extra)
    cmd.extend(packages)
    subprocess.check_call(cmd)

def pin_hf():
    pip(*HF_STACK, extra=["--force-reinstall"])

def ensure_hf_compat():
    """Patch transformers for coqui-tts on Colab."""
    import torch
    import transformers.pytorch_utils as pytorch_utils
    import transformers.utils.import_utils as import_utils

    if not hasattr(pytorch_utils, "isin_mps_friendly"):
        pytorch_utils.isin_mps_friendly = torch.isin

    if not hasattr(import_utils, "is_torch_greater_or_equal"):
        def is_torch_greater_or_equal(version_str, /):
            cur = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:3])
            tgt = tuple(int(x) for x in version_str.split(".")[:3])
            while len(cur) < 3:
                cur += (0,)
            while len(tgt) < 3:
                tgt += (0,)
            return cur >= tgt
        import_utils.is_torch_greater_or_equal = is_torch_greater_or_equal

# 1) numpy 2.x + scipy
pip("numpy>=2.0,<2.3", "scipy>=1.12.0,<2.0.0", extra=["--upgrade"])

# 2) transformers + tokenizers (must match)
pin_hf()

# 3) orchestration
pip(
    "langgraph>=0.2.0,<0.3.0",
    "langgraph-checkpoint-sqlite>=2.0.0,<3.0.0",
    "tenacity",
    "bitsandbytes",
    "accelerate",
    "duckduckgo-search",
    "pydub",
    "tqdm",
)

# 4) coqui TTS, then re-pin HF stack
pip("coqui-tts>=0.24.0,<0.28.0")
pin_hf()

# 5) audiocraft without touching Colab torch
pip("av>=12.0.0")
pip("audiocraft==1.3.0", extra=["--no-deps"])
pip("encodec", "flashy", "num2words", "hydra-core", "hydra-colorlog", "torchmetrics", "demucs", "einops")

# 6) final pins
pin_hf()
pip("numpy>=2.0,<2.3", "scipy>=1.12.0,<2.0.0", extra=["--upgrade"])

!apt-get install -qq ffmpeg

import numpy as np
import scipy
import tokenizers
import torch
import transformers

ensure_hf_compat()

from TTS.api import TTS
from audiocraft.models import MusicGen
import langgraph

print("numpy", np.__version__)
print("scipy", scipy.__version__)
print("tokenizers", tokenizers.__version__)
print("transformers", transformers.__version__)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
print("All dependencies installed successfully!")

In [ ]:
import os

if os.path.isdir("PodcastGen-Agent"):
    %cd PodcastGen-Agent
    !git pull
else:
    !git clone https://github.com/Rashal10/PodcastGen-Agent.git
    %cd PodcastGen-Agent

!pip install -q -e . --no-deps
!pip install -q --force-reinstall "transformers==4.46.3" "tokenizers==0.20.3"

import tokenizers, transformers
print("tokenizers", tokenizers.__version__)
print("transformers", transformers.__version__)
print("Repo ready")

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import os

os.environ["COQUI_TOS_AGREED"] = "1"
os.environ["PODCAST_LOG_LEVEL"] = "INFO"

TOPIC = "LoRA and QLoRA"
DURATION = 5

!python -m podcast_gen_agent.main "{TOPIC}" --duration {DURATION}

In [ ]:
from IPython.display import Audio, display
from pathlib import Path

from podcast_gen_agent.utils.output import find_latest_podcast

latest = find_latest_podcast(Path("outputs"))
if latest:
    print(f"Playing: {latest}")
    display(Audio(str(latest)))
else:
    print("No podcast mp3 found. Check the generation cell for errors.")

In [ ]:
from pathlib import Path
from google.colab import files

from podcast_gen_agent.utils.output import find_latest_podcast

latest = find_latest_podcast(Path("outputs"))
if latest:
    print(f"Downloading: {latest}")
    files.download(str(latest))
else:
    print("No podcast file found under outputs/. Check the generation cell for errors.")